In [1]:
# !pip uninstall gensim --yes
# !pip install gensim==3.8.1
# !pip install deeprobust==0.2.9
# !pip install torch_geometric
# !pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.2.0+cpu.html

In [2]:
from deeprobust.graph.data import Dpr2Pyg, Pyg2Dpr
from deeprobust.graph.data import Dataset as DRDataset
import torch
from torch_geometric.data import Data
import numpy as np
from deeprobust.graph.data import Dataset, PrePtbDataset, PtbDataset
from deeprobust.graph.defense import GCN, RGCN, ProGNN, SimPGCN, GCNSVD, GCNJaccard
from deeprobust.graph.global_attack import Metattack, DICE, Random, PGDAttack
from scipy.sparse import csr_matrix
import torch
from torch.nn import Linear
import torch.nn.functional as F
from torch_geometric.transforms import Compose
from torch_geometric.datasets import Amazon
from torch_geometric.transforms.random_node_split import RandomNodeSplit
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures
from torch_geometric.nn import GCNConv
from torch_geometric.nn import GATConv
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import SAGEConv
from sklearn.metrics import roc_auc_score

from torch_geometric.utils import negative_sampling
from torch_geometric.utils import train_test_split_edges

from copy import deepcopy
import torch.nn as nn
from IPython.display import Javascript  # Restrict height of output cell.
import matplotlib.pyplot as plt

from GSage import GSAGE
from GSaint import GSAINT
from GAT import GAT

In [5]:
seed = 15
ptb_rates = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3]
# ptb_rates = [0.025, 0.05]
# ptb_rate = 0.25
# ptb_rate1 = 0.15
dataset = 'citeseer'
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

In [6]:
data = DRDataset(root='/tmp/', name=dataset, setting='prognn')
adj, features, labels = data.adj, data.features, data.labels
idx_train, idx_val, idx_test = data.idx_train, data.idx_val, data.idx_test
idx_unlabeled = np.union1d(idx_val, idx_test)
idx_unlabeled = np.union1d(idx_val, idx_test)


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# budget = int(ptb_rate * (adj.todense().sum() // 2))
# print(budget)
# budget1 = int(ptb_rate1 * (adj.todense().sum() // 2))
# print(budget1)

Loading citeseer dataset...
Selecting 1 largest connected components
Dowloading from https://raw.githubusercontent.com/ChandlerBang/Pro-GNN/master/splits/citeseer_prognn_splits.json to /tmp/citeseer_prognn_splits.json


In [7]:
np.save(f'/tmp/{dataset}_adj.npy',adj.todense())

In [8]:
np.save(f'/tmp/{dataset}_features.npy',features.todense())

In [9]:
np.save(f'/tmp/{dataset}_labels.npy',labels)

In [10]:
np.save(f'/tmp/{dataset}_idx_test',idx_test)

# Metattack

## GAT

In [11]:
surrogate5 = GAT(nfeat=features.shape[1],
      nhid=8, heads=8,
      nclass=labels.max().item() + 1,
      dropout=0.5, device=device)
surrogate5 = surrogate5.to(device)

pyg_data = Dpr2Pyg(data)
surrogate5.fit(pyg_data, verbose=True) # train with earlystopping
surrogate5.test()

Processing...
/home/wcorcoran/anaconda3/envs/mesp/lib/python3.8/site-packages/deeprobust/graph/data/pyg_dataset.py:48: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:245.)
  edge_index = torch.LongTensor(dpr_data.adj.nonzero())
Done!
/home/wcorcoran/anaconda3/envs/mesp/lib/python3.8/site-packages/torch_geometric/data/in_memory_dataset.py:157: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


=== training GAT model ===
Epoch 0, training loss: 1.8220605850219727
Epoch 10, training loss: 0.5069877505302429
Epoch 20, training loss: 0.3469322621822357
Epoch 30, training loss: 0.310787558555603
Epoch 40, training loss: 0.28650763630867004
Epoch 50, training loss: 0.39581498503685
Epoch 60, training loss: 0.23617719113826752
Epoch 70, training loss: 0.2284516990184784
Epoch 80, training loss: 0.31404373049736023
Epoch 90, training loss: 0.33287879824638367
Epoch 100, training loss: 0.29722070693969727
=== early stopping at 106, loss_val = 0.840660572052002 ===
Test set results: loss= 0.8618 accuracy= 0.7305


0.7304502369668247

In [12]:
surrogate5.eval()
preds=surrogate5.predict()
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total
benchmark_clean = test_accuracy

print(f'Test Accuracy: {test_accuracy}')

Test Accuracy: 0.7304502369668247


In [13]:
from copy import deepcopy

# ptb_rates = [0.1, 0.2, 0.3, 0.4, 0.5]
gat_results = []

for ptb in ptb_rates:
  budget = int(ptb * (adj.todense().sum() // 2))
  # Setup Attack Model
  model = Metattack(surrogate5, nnodes=adj.shape[0], feature_shape=features.shape,
          attack_structure=True, attack_features=False, device=device, lambda_=0).to(device)
  # Attack
  model.attack(features, adj, labels, idx_train, idx_unlabeled, n_perturbations=budget, ll_constraint=False)
  modified_adj = model.modified_adj # modified_adj is a torch.tensor
  modified_adj = modified_adj.cpu().numpy()
  modified_adj = csr_matrix(modified_adj)

  perturbed_adj = deepcopy(modified_adj)

  data_copied = deepcopy(data)
  data_copied.adj = modified_adj

  atk_model = GAT(nfeat=features.shape[1],
        nhid=8, heads=8,
        nclass=labels.max().item() + 1,
        dropout=0.5, device=device)
  atk_model = surrogate5.to(device)
  atk_model.fit(Dpr2Pyg(data_copied), patience=100, verbose=True)

  atk_acc = atk_model.test()

  print("accuracy: ", atk_acc)
  print("benchmark change: ", atk_acc - benchmark_clean)
  gat_results.append(atk_acc - benchmark_clean)

Perturbing graph:   0%|          | 0/183 [00:00<?, ?it/s]

GCN loss on unlabled data: 1.1803910732269287
GCN acc on unlabled data: 0.7030015797788309
attack loss: 0.47382456064224243


Perturbing graph:   1%|          | 1/183 [00:01<03:09,  1.04s/it]

GCN loss on unlabled data: 1.2260735034942627
GCN acc on unlabled data: 0.6998420221169036
attack loss: 0.46440568566322327


Perturbing graph:   1%|          | 2/183 [00:02<03:02,  1.01s/it]

...

GCN loss on unlabled data: 3.3255701065063477
GCN acc on unlabled data: 0.44023170089520797
attack loss: 5.257898330688477


Perturbing graph: 100%|█████████▉| 1099/1100 [24:44<00:01,  1.37s/it]

GCN loss on unlabled data: 2.892732620239258
GCN acc on unlabled data: 0.46866771985255395
attack loss: 4.629292964935303


Perturbing graph: 100%|██████████| 1100/1100 [24:46<00:00,  1.35s/it]
Processing...
Done!


=== training GAT model ===
Epoch 0, training loss: 1.806922435760498
Epoch 10, training loss: 0.5881398320198059
Epoch 20, training loss: 0.35436180233955383
Epoch 30, training loss: 0.25000593066215515
Epoch 40, training loss: 0.20047080516815186
Epoch 50, training loss: 0.23924291133880615
Epoch 60, training loss: 0.22764907777309418
Epoch 70, training loss: 0.2316620647907257
Epoch 80, training loss: 0.12538747489452362
Epoch 90, training loss: 0.1447944939136505
Epoch 100, training loss: 0.18061143159866333
=== early stopping at 104, loss_val = 1.2337864637374878 ===
Test set results: loss= 1.2243 accuracy= 0.6297
accuracy:  0.629739336492891
benchmark change:  -0.1007109004739336


## GSAINT

In [14]:
# Setup Surrogate model
surrogate_saint = GSAINT(nfeat=features.shape[1], nhid=64, nclass=labels.max().item()+1, dropout=0, with_bias=True, device=device).to(device)
surrogate_saint.fit(data, patience=100, verbose=True)

Processing...
Done!
Compute GraphSAINT normalization: : 220562it [00:00, 793905.62it/s]                          


=== training GSAINT model ===
Epoch 0, training loss: 0.02857412025332451
Epoch 10, training loss: 0.0010273581137880683
Epoch 20, training loss: 0.0009432145161554217
Epoch 30, training loss: 0.0009101084433495998
Epoch 40, training loss: 0.0009507921058684587
Epoch 50, training loss: 0.0009478987194597721
Epoch 60, training loss: 0.0009180554188787937
Epoch 70, training loss: 0.0009194521699100733
Epoch 80, training loss: 0.0009361926931887865
Epoch 90, training loss: 0.0009085937053896487
Epoch 100, training loss: 0.0009758258238434792
=== early stopping at 109, loss_val = 0.6946503520011902 ===


In [15]:
preds=surrogate_saint.predict()
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

Test Accuracy: 0.7511848341232228


In [16]:
benchmark_clean = test_accuracy

In [17]:
from copy import deepcopy

gsaint_results = []

for ptb in ptb_rates:
  torch.cuda.empty_cache()
  budget = int(ptb * (adj.todense().sum() // 2))
  # Setup Attack Model
  model = Metattack(surrogate_saint, nnodes=adj.shape[0], feature_shape=features.shape,
          attack_structure=True, attack_features=False, device=device, lambda_=0).to(device)
  # Attack
  model.attack(features, adj, labels, idx_train, idx_unlabeled, n_perturbations=budget, ll_constraint=False)
  modified_adj = model.modified_adj # modified_adj is a torch.tensor
  modified_adj = modified_adj.cpu().numpy()
  modified_adj = csr_matrix(modified_adj)

  perturbed_adj = deepcopy(modified_adj)
  data_copied = deepcopy(data)
  data_copied.adj = modified_adj
  atk_model = GSAINT(nfeat=features.shape[1], nhid=64, nclass=labels.max().item()+1, dropout=0, with_bias=True, device=device).to(device)
  atk_model.fit(data_copied, patience=100, verbose=True)

  atk_acc = atk_model.test()
  print("accuracy: ", atk_acc)
  print("benchmark change: ", atk_acc - benchmark_clean)
  gsaint_results.append(atk_acc - benchmark_clean)


Perturbing graph:   0%|          | 0/183 [00:00<?, ?it/s]

GCN loss on unlabled data: 1.1238508224487305
GCN acc on unlabled data: 0.7261716692996313
attack loss: 0.3387753665447235


Perturbing graph:   1%|          | 1/183 [00:01<04:14,  1.40s/it]

GCN loss on unlabled data: 1.1176846027374268
GCN acc on unlabled data: 0.723012111637704
attack loss: 0.3345676362514496


Perturbing graph:   1%|          | 2/183 [00:02<04:10,  1.38s/it]

...

GCN loss on unlabled data: 6.3647780418396
GCN acc on unlabled data: 0.2912058978409689
attack loss: 6.118871688842773


Perturbing graph: 100%|█████████▉| 1099/1100 [25:44<00:01,  1.38s/it]

GCN loss on unlabled data: 6.486642360687256
GCN acc on unlabled data: 0.29331226961558715
attack loss: 6.2538628578186035


Perturbing graph: 100%|██████████| 1100/1100 [25:45<00:00,  1.41s/it]
Processing...
Done!
Compute GraphSAINT normalization: : 220899it [00:00, 605755.09it/s]                          


=== training GSAINT model ===
Epoch 0, training loss: 0.04774017632007599
Epoch 10, training loss: 0.0011627224739640951
Epoch 20, training loss: 0.0010983298998326063
Epoch 30, training loss: 0.0011313077993690968
Epoch 40, training loss: 0.0010933834128081799
Epoch 50, training loss: 0.001058851950801909
Epoch 60, training loss: 0.0010012183338403702
Epoch 70, training loss: 0.0010859339963644743
Epoch 80, training loss: 0.20287413895130157
Epoch 90, training loss: 0.0008851552847772837
Epoch 100, training loss: 0.001003714045509696
Epoch 110, training loss: 0.0010604874696582556
Epoch 120, training loss: 0.0010633289348334074
Epoch 130, training loss: 0.0010830445680767298
=== early stopping at 137, loss_val = 0.8354393243789673 ===
accuracy:  0.6783175355450237
benchmark change:  -0.07286729857819907


## GSAGE

In [ ]:
# Setup Surrogate model
surrogate_sage = GSAGE(nfeat=features.shape[1], nhid=64, nclass=labels.max().item()+1, dropout=0, with_bias=True, device=device).to(device)
surrogate_sage.fit(data, patience=100, verbose=True)

In [ ]:
preds=surrogate_sage.predict()
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

In [ ]:
benchmark_clean = test_accuracy

In [ ]:
from copy import deepcopy

gsage_results = []

for ptb in ptb_rates:
  torch.cuda.empty_cache()
  budget = int(ptb * (adj.todense().sum() // 2))
  # Setup Attack Model
  model = Metattack(surrogate_sage, nnodes=adj.shape[0], feature_shape=features.shape,
          attack_structure=True, attack_features=False, device=device, lambda_=0).to(device)
  # Attack
  model.attack(features, adj, labels, idx_train, idx_unlabeled, n_perturbations=budget, ll_constraint=False)
  modified_adj = model.modified_adj # modified_adj is a torch.tensor
  modified_adj = modified_adj.cpu().numpy()
  modified_adj = csr_matrix(modified_adj)

  perturbed_adj = deepcopy(modified_adj)
  data_copied = deepcopy(data)
  data_copied.adj = modified_adj
  atk_model = GSAGE(nfeat=features.shape[1], nhid=64, nclass=labels.max().item()+1, dropout=0, with_bias=True, device=device).to(device)
  atk_model.fit(data_copied, patience=100, verbose=True)

  atk_acc = atk_model.test()
  print("accuracy: ", atk_acc)
  print("benchmark change: ", atk_acc - benchmark_clean)
  gsage_results.append(atk_acc - benchmark_clean)


## GCN

In [ ]:
# Setup Surrogate model
surrogate = GCN(nfeat=features.shape[1], nclass=labels.max().item()+1,
                nhid=16, dropout=0, with_relu=True, with_bias=True, device=device).to(device)
surrogate.fit(features, adj, labels, idx_train, idx_val, patience=100, verbose=True)

In [ ]:
preds=surrogate.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

In [ ]:
benchmark_clean = test_accuracy

In [ ]:
from copy import deepcopy

gcn_results = []

for ptb in ptb_rates:
  torch.cuda.empty_cache()
  budget = int(ptb * (adj.todense().sum() // 2))
  # Setup Attack Model
  model = Metattack(surrogate, nnodes=adj.shape[0], feature_shape=features.shape,
          attack_structure=True, attack_features=False, device=device, lambda_=0).to(device)
  # Attack
  model.attack(features, adj, labels, idx_train, idx_unlabeled, n_perturbations=budget, ll_constraint=False)
  modified_adj = model.modified_adj # modified_adj is a torch.tensor
  modified_adj = modified_adj.cpu().numpy()
  modified_adj = csr_matrix(modified_adj)

  perturbed_adj = deepcopy(modified_adj)
  atk_model = GCN(nfeat=features.shape[1], nclass=labels.max().item()+1,
                nhid=16, dropout=0, with_relu=True, with_bias=True, device=device).to(device)
  atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

  atk_acc = atk_model.test(idx_test)
  print("accuracy: ", atk_acc)
  print("benchmark change: ", atk_acc - benchmark_clean)
  gcn_results.append(atk_acc - benchmark_clean)


In [ ]:
gcn_results

## GCNJaccard

In [ ]:
surrogate1 = GCNJaccard(nfeat=features.shape[1],
          nhid=16,
          nclass=labels.max().item() + 1,
          dropout=0.5, device=device).to(device)
surrogate1.fit(features, adj, labels, idx_train, idx_val, threshold=0.03)

In [ ]:
preds=surrogate1.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total
benchmark_clean = test_accuracy

print(f'Test Accuracy: {test_accuracy}')

In [ ]:
from copy import deepcopy

# ptb_rates = [0.1, 0.2, 0.3, 0.4, 0.5]
jaccard_results = []

for ptb in ptb_rates:
  budget = int(ptb * (adj.todense().sum() // 2))
  # Setup Attack Model
  model = Metattack(surrogate1, nnodes=adj.shape[0], feature_shape=features.shape,
          attack_structure=True, attack_features=False, device=device, lambda_=0).to(device)
  # Attack
  model.attack(features, adj, labels, idx_train, idx_unlabeled, n_perturbations=budget, ll_constraint=False)
  modified_adj = model.modified_adj # modified_adj is a torch.tensor
  modified_adj = modified_adj.cpu().numpy()
  modified_adj = csr_matrix(modified_adj)

  perturbed_adj = deepcopy(modified_adj)
  atk_model = GCNJaccard(nfeat=features.shape[1],
            nhid=16,
            nclass=labels.max().item() + 1,
            dropout=0.5, device=device).to(device)
  atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

  atk_acc = atk_model.test(idx_test)

  print("accuracy: ", atk_acc)
  print("benchmark change: ", atk_acc - benchmark_clean)
  jaccard_results.append(atk_acc - benchmark_clean)


## Plotting

In [ ]:
# Plotting
plt.plot(ptb_rates, gcn_results, label="GCN")
plt.plot(ptb_rates, jaccard_results, label="Jaccard")
plt.plot(ptb_rates, gat_results, label="GAT")
plt.plot(ptb_rates, gsage_results, label="GSAGE")
plt.plot(ptb_rates, gsaint_results, label="GSAINT")

plt.xlabel('PTB Rates')
plt.ylabel('Change in Accuracy (abs. value)')
plt.title(f'Attacks on {dataset} Dataset')
plt.legend()

# Show plot
plt.show()

output_path = f"./data/{dataset}_graph.png"
plt.savefig(output_path)